[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap08_machine_learning/agrupamento_caracteristicas_compra_caso_tech.ipynb)

# Agrupamento por características de compra

Este notebook explora agrupamentos de pedidos da Base didática Caso Tech. Define a unidade de análise, agrega atributos comerciais, padroniza as escalas e aplica duas técnicas: K-means e agrupamento hierárquico aglomerativo. A análise examina a estabilidade e a interpretação dos grupos, sem tratá-los como categorias naturais comprovadas.

O botão pressupõe que o notebook seja publicado em `capitulos/cap08_machine_learning/agrupamento_caracteristicas_compra_caso_tech.ipynb` no repositório do livro.

## 1. Objetivo e cuidados de interpretação

O agrupamento é uma técnica não supervisionada: não há uma variável-alvo que indique previamente a qual grupo cada observação pertence. O algoritmo organiza as observações conforme uma medida de similaridade e as variáveis selecionadas.

Neste exercício, cada observação será um pedido, e os atributos serão agregados por pedido. As soluções obtidas dependem da escolha das variáveis, da escala, da métrica de distância e dos parâmetros dos algoritmos. Assim, os grupos são descrições exploratórias úteis para organizar a análise, não categorias naturais ou segmentos comprovados da população.

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram

SEMENTE = 2026
URL_BASE = "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"
np.random.seed(SEMENTE)

## 2. Carregamento e inspeção da base

O notebook tenta carregar a planilha do repositório do projeto. Se isso não for possível no Colab, permite enviar o arquivo manualmente. Inspecione as abas e colunas antes de configurar as variáveis de análise; os nomes podem variar conforme a versão da base.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE, timeout=30) as resposta:
            return pd.ExcelFile(io.BytesIO(resposta.read()))
    except Exception as erro:
        print("Download indisponível:", erro)
        try:
            from google.colab import files
            enviados = files.upload()
            if not enviados:
                raise RuntimeError("Nenhum arquivo selecionado.")
            return pd.ExcelFile(io.BytesIO(enviados[next(iter(enviados))]))
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha localmente.")

excel=carregar_excel()
bases={aba:pd.read_excel(excel,sheet_name=aba) for aba in excel.sheet_names}
for aba,df in bases.items():
    print(f"\n{aba}: {df.shape[0]} linhas x {df.shape[1]} colunas")
    print(list(df.columns))

## 3. Definição da unidade de análise e agregação

A unidade escolhida é o pedido. A base Caso Tech pode conter mais de uma linha por pedido, portanto não se deve somar diretamente valores repetidos sem verificar a estrutura. Configure abaixo os nomes exatos das colunas e defina a regra de agregação de cada atributo.

Exemplos de atributos comerciais potencialmente úteis são valor total do pedido, quantidade de itens, número de linhas ou categorias de produto e participação de cada categoria. Use somente variáveis com significado coerente na granularidade escolhida. Se o valor total do pedido estiver repetido em cada linha, use uma regra como primeiro valor após verificar que ele é consistente dentro do pedido, não a soma das repetições.

In [ ]:
# Configure com os nomes exatos observados na inspeção.
ABA = None
COLUNA_PEDIDO = None
COLUNA_VALOR_PEDIDO = None  # valor total, se disponível e consistente dentro do pedido
COLUNA_QUANTIDADE = None   # quantidade por linha, se disponível
COLUNA_CATEGORIA = None    # categoria de produto, opcional
COLUNA_LOJA = None         # loja, opcional
COLUNA_CANAL = None        # canal, opcional

if not ABA or ABA not in bases or not COLUNA_PEDIDO:
    raise ValueError("Informe ABA e COLUNA_PEDIDO com os nomes exatos da base.")
df=bases[ABA].copy()
if COLUNA_PEDIDO not in df.columns:
    raise KeyError("A coluna de pedido informada não foi encontrada.")
print("Pedidos distintos:",df[COLUNA_PEDIDO].nunique())
print("Linhas:",len(df))
print("Duplicidade de pedido:",df[COLUNA_PEDIDO].duplicated().sum())

# Conferência das colunas numéricas repetidas: não agregar antes de examinar esta tabela.
colunas_conferir=[c for c in [COLUNA_VALOR_PEDIDO] if c and c in df.columns]
if colunas_conferir:
    display(df.groupby(COLUNA_PEDIDO)[colunas_conferir].nunique(dropna=True).describe())

# Agrega uma linha por pedido. Ajuste os campos conforme a semântica e a estrutura da base.
agg={}
if COLUNA_VALOR_PEDIDO:
    agg[COLUNA_VALOR_PEDIDO]="first"  # usar somente após confirmar consistência no pedido
if COLUNA_QUANTIDADE:
    agg[COLUNA_QUANTIDADE]="sum"     # quantidade de linha, agregável por soma
if COLUNA_CATEGORIA:
    agg[COLUNA_CATEGORIA]="nunique"   # diversidade de categorias
if COLUNA_LOJA:
    agg[COLUNA_LOJA]="nunique"
if COLUNA_CANAL:
    agg[COLUNA_CANAL]="first"
if not agg:
    raise ValueError("Configure pelo menos um atributo comercial para agregação.")

pedidos=df.groupby(COLUNA_PEDIDO).agg(agg).reset_index()
pedidos=pedidos
print("Unidade de análise: pedido")
print("Tabela agregada:",pedidos.shape)
display(pedidos.head())

## 4. Construção dos atributos e validação

A tabela agregada deve ser revisada antes do agrupamento. Remova a chave do pedido, identificadores e atributos que não expressem características de compra comparáveis. Campos categóricos nominais não devem ser convertidos arbitrariamente em números. Neste exemplo, as categorias podem ser representadas por indicadores de frequência por pedido, se a análise exigir essa informação; mantenha a representação e a interpretação explícitas.

A célula seguinte usa os atributos numéricos agregados configurados. Para incluir uma variável categórica, crie previamente indicadores apropriados ou uma proporção por categoria, garantindo que a soma e o denominador tenham interpretação clara.

In [ ]:
# Seleção manual dos atributos numéricos agregados que entrarão no agrupamento.
ATRIBUTOS = [c for c in pedidos.columns if c != COLUNA_PEDIDO and pd.api.types.is_numeric_dtype(pedidos[c])]
# Revise a lista. Exclua IDs, códigos e variáveis sem significado quantitativo.
print("Atributos candidatos:",ATRIBUTOS)
# Exemplo de ajuste manual: ATRIBUTOS = ["valor total", "quantidade", "diversidade categorias"]
if len(ATRIBUTOS)<2:
    raise ValueError("Selecione pelo menos dois atributos quantitativos adequados.")

X_bruto=pedidos[ATRIBUTOS].replace([np.inf,-np.inf],np.nan).copy()
# Exclui atributos completamente ausentes ou constantes.
validas=[c for c in X_bruto.columns if X_bruto[c].notna().any() and X_bruto[c].nunique(dropna=True)>1]
X_bruto=X_bruto[validas]
ATRIBUTOS=validas
if len(ATRIBUTOS)<2:
    raise ValueError("Menos de dois atributos variáveis após a validação.")
print("Atributos finais:",ATRIBUTOS)
display(X_bruto.describe().T)

## 5. Tratamento de dados e escalas

A distância entre observações é sensível às unidades de medida. Valores monetários podem dominar contagens ou medidas de diversidade se forem mantidos em escalas muito diferentes. O notebook imputa valores ausentes pela mediana e padroniza os atributos, usando média e desvio-padrão estimados na tabela de análise. A padronização não elimina assimetria nem valores extremos; examine as distribuições e avalie transformações somente quando justificadas pelo significado dos dados.

In [ ]:
imputador=SimpleImputer(strategy="median")
X_imputado=imputador.fit_transform(X_bruto)
escalador=StandardScaler()
X_padronizado=escalador.fit_transform(X_imputado)
X_padronizado=pd.DataFrame(X_padronizado,columns=ATRIBUTOS,index=pedidos.index)
print("Médias padronizadas (aprox.):")
display(X_padronizado.mean().round(3).to_frame("média"))
print("Desvios-padrão padronizados (aprox.):")
display(X_padronizado.std(ddof=0).round(3).to_frame("desvio padrão"))

## 6. Número de grupos e técnicas de agrupamento

São aplicadas duas técnicas complementares:

1. **K-means**, método particional que representa cada grupo por um centroide e busca reduzir a soma das distâncias quadráticas intragrupo. É sensível à escala e à inicialização; por isso são usadas múltiplas inicializações e semente fixa.
2. **Agrupamento hierárquico aglomerativo**, que começa com observações separadas e as une progressivamente. Usa-se ligação de Ward, que procura minimizar o aumento da variância intragrupo em cada fusão. A técnica gera uma hierarquia e permite comparar partições com diferentes números de grupos.

O coeficiente de silhueta é usado como diagnóstico descritivo para diferentes valores de k. Ele não determina sozinho a solução substantivamente mais adequada.

In [ ]:
n=len(X_padronizado)
if n<10:
    raise ValueError("Poucas observações para esta comparação didática.")
ks=list(range(2,min(8,n-1)+1))
linhas=[]
for k in ks:
    km=KMeans(n_clusters=k,n_init=20,random_state=SEMENTE)
    rot=km.fit_predict(X_padronizado)
    sil=silhouette_score(X_padronizado,rot)
    linhas.append({"k":k,"inércia K-means":km.inertia_,"silhueta K-means":sil})

hier_resultados={}
for k in ks:
    hier=AgglomerativeClustering(n_clusters=k,linkage="ward")
    rot=hier.fit_predict(X_padronizado)
    hier_resultados[k]=rot
    linhas[k-2]["silhueta hierárquica"] = silhouette_score(X_padronizado,rot)

diagnosticos=pd.DataFrame(linhas).set_index("k")
display(diagnosticos.round(3))
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(diagnosticos.index,diagnosticos["silhueta K-means"],marker="o",label="K-means")
ax.plot(diagnosticos.index,diagnosticos["silhueta hierárquica"],marker="s",label="Hierárquico, Ward")
ax.set_xlabel("Número de grupos (k)");ax.set_ylabel("Silhueta média")
ax.set_title("Diagnóstico de separação para diferentes k");ax.legend();plt.show()

## 7. Dendrograma hierárquico

O dendrograma representa as fusões sucessivas do agrupamento hierárquico. Para evitar uma figura ilegível em bases grandes, o gráfico é produzido com uma amostra reprodutível de até 300 pedidos. A visualização é ilustrativa; a partição escolhida e os resultados devem ser calculados na base analítica completa.

In [ ]:
limite=300
if n>limite:
    indices=np.random.default_rng(SEMENTE).choice(n,size=limite,replace=False)
    X_dendro=X_padronizado.iloc[indices]
else:
    X_dendro=X_padronizado
Z=linkage(X_dendro,method="ward")
plt.figure(figsize=(12,5))
dendrogram(Z,no_labels=True,truncate_mode="lastp",p=30)
plt.title("Dendrograma resumido: agrupamento hierárquico de Ward")
plt.xlabel("Observações ou grupos resumidos")
plt.ylabel("Distância de fusão")
plt.tight_layout();plt.show()

## 8. Seleção exploratória da solução e estabilidade

Para comparar estabilidade, o notebook repete o K-means com diferentes sementes e calcula o índice Rand ajustado entre as partições. Valores mais altos indicam maior concordância entre duas partições, descontada a concordância esperada ao acaso. A estabilidade é avaliada para um k escolhido pelo pesquisador após examinar diagnósticos, tamanhos de grupo e interpretabilidade. Não se deve selecionar k apenas porque maximiza uma métrica.

In [ ]:
# Defina k após examinar os diagnósticos e considerar a utilidade interpretativa.
K_ESCOLHIDO = None
if K_ESCOLHIDO is None or K_ESCOLHIDO not in ks:
    raise ValueError("Defina K_ESCOLHIDO como um dos valores avaliados, com justificativa.")

# Comparação entre várias inicializações-semente no K-means.
sementes=[11,23,37,51,2026]
particoes=[]
for s in sementes:
    modelo=KMeans(n_clusters=K_ESCOLHIDO,n_init=20,random_state=s)
    particoes.append(modelo.fit_predict(X_padronizado))
acordos=[]
for i in range(len(particoes)):
    for j in range(i+1,len(particoes)):
        acordos.append(adjusted_rand_score(particoes[i],particoes[j]))
print("ARI médio entre sementes:",round(float(np.mean(acordos)),3))
print("ARI mínimo entre sementes:",round(float(np.min(acordos)),3))

# Escolha uma partição reprodutível final. Compare também com Ward no mesmo k.
modelo_kmeans=KMeans(n_clusters=K_ESCOLHIDO,n_init=20,random_state=SEMENTE)
rotulos_kmeans=modelo_kmeans.fit_predict(X_padronizado)
rotulos_ward=AgglomerativeClustering(n_clusters=K_ESCOLHIDO,linkage="ward").fit_predict(X_padronizado)
print("ARI entre K-means e Ward:",round(adjusted_rand_score(rotulos_kmeans,rotulos_ward),3))

## 9. Tamanhos e interpretação dos grupos

A interpretação é feita por meio de estatísticas descritivas dos atributos originais, antes da padronização, e dos tamanhos dos grupos. Os grupos recebem rótulos numéricos apenas para identificação. A descrição deve evitar atribuir causas ou tratar os agrupamentos como segmentos naturais confirmados. Se forem nomeados de forma interpretativa, os nomes devem resumir padrões observados e ser apresentados como conveniência analítica.

In [ ]:
resultado=pedidos[[COLUNA_PEDIDO]+ATRIBUTOS].copy()
resultado["grupo_kmeans"]=rotulos_kmeans
resultado["grupo_ward"]=rotulos_ward
print("Tamanho dos grupos K-means:")
display(resultado["grupo_kmeans"].value_counts().sort_index().to_frame("pedidos"))
print("Resumo por grupo K-means nos atributos originais:")
display(resultado.groupby("grupo_kmeans")[ATRIBUTOS].agg(["count","mean","median","std"]).round(2))
print("\nTamanho dos grupos Ward:")
display(resultado["grupo_ward"].value_counts().sort_index().to_frame("pedidos"))
print("Resumo por grupo Ward nos atributos originais:")
display(resultado.groupby("grupo_ward")[ATRIBUTOS].median().round(2))

## 10. Conclusão e limites

K-means e agrupamento hierárquico de Ward produzem partições a partir das variáveis e da escala definidas pelo pesquisador. A comparação de silhueta, tamanhos, perfis e estabilidade ajuda a avaliar se a solução é útil para fins exploratórios. Resultados que variam muito entre inicializações ou entre métodos devem ser interpretados com cautela.

A análise não demonstra a existência de categorias naturais de clientes ou pedidos. Os grupos são dependentes das escolhas metodológicas e dos atributos agregados. A estabilidade computacional também não equivale à validação substantiva ou à estabilidade em outras amostras e períodos. Uma aplicação gerencial exigiria validação com conhecimento do processo, verificação em períodos distintos e avaliação de utilidade para a decisão pretendida.

### Checklist

- A unidade de análise está definida como pedido?
- A agregação evita somar valores repetidos por pedido?
- Os atributos representam características de compra e não identificadores?
- As escalas foram tratadas antes do cálculo de distâncias?
- Foram comparados K-means e agrupamento hierárquico de Ward?
- A escolha de k considerou diagnósticos, tamanhos e interpretação?
- A estabilidade foi examinada sem confundi-la com prova de categorias naturais?
- A descrição dos grupos permanece exploratória e não causal?